In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
#import geopandas
import tensorflow
import keras
from keras.models import Model
from keras.layers import LSTM, Activation, Dense, Dropout, Input, Embedding
from keras.optimizers import RMSprop
#from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing import sequence
#from keras.utils import to_categorical
from keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
import numpy as np
import math
import sklearn.metrics
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project='geewildfires')

T4 GPU works well without costing a lot, L100 faster but expensive


In [ ]:
def open_file(gcs_path):
    full_file = 'gs://jk_wildfiresspreadts/' + gcs_path
    df = pd.read_csv(full_file)
    return df

In [ ]:
fires_df = open_file("VALIDATION_PartFourOutput_WUI.csv")
fires_df.head()

,Unnamed: 0,FIRE_ID,UrbanAngle,occur_id,point_id,IsUrban,WUIBreach,Month,NDVI,NDWI,NBR,DEM,aspect,hillshade,slope,bi,erc,eto,fm100,fm1000,pr,rmax,rmin,th,pop_density,tmmn,tmmx,vpd,vs,LandCover,Y,Water,Very_low_rural,Low_density_rural,Rural_cluster,Suburban,Semi_dense_urban,Dense_urban,Urban_centre,first_urban_point,RemoveFlag
0,0,CA3622612010420250902,-163.647115,1,0,0,0,9,0.141225,0.178077,0.019172,102.352814,56,180,0,62.0,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,0.0,293.0,313.5,3.75,2.7,81,1,0,1,0,0,0,0,0,0,60,False
1,1,CA3622612010420250902,-163.647115,1,1,0,0,9,0.111281,0.156092,-0.058161,102.741325,19,180,0,62.0,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,0.0,293.0,313.5,3.75,2.7,81,1,0,1,0,0,0,0,0,0,60,False
2,2,CA3622612010420250902,-163.647115,1,2,0,0,9,0.130974,0.176366,-0.006227,102.741325,19,180,0,62.0,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,0.0,293.0,313.5,3.75,2.7,82,1,0,1,0,0,0,0,0,0,60,False
3,3,CA3622612010420250902,-163.647115,1,3,0,0,9,0.131013,0.176502,0.007718,103.105774,51,179,0,62.0,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,0.0,293.0,313.5,3.75,2.7,82,1,0,1,0,0,0,0,0,0,60,False
4,4,CA3622612010420250902,-163.647115,1,4,0,0,9,0.099985,0.143464,-0.029291,103.281387,341,181,0,62.0,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,0.0,293.0,313.5,3.75,2.7,82,1,0,1,0,0,0,0,0,0,60,False


In [ ]:
print(fires_df.info(verbose=True, show_counts=True))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1982 entries, 0 to 1981
Data columns (total 41 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         1982 non-null   int64  
 1   FIRE_ID            1982 non-null   object 
 2   UrbanAngle         1982 non-null   float64
 3   occur_id           1982 non-null   int64  
 4   point_id           1982 non-null   int64  
 5   IsUrban            1982 non-null   int64  
 6   WUIBreach          1982 non-null   int64  
 7   Month              1982 non-null   int64  
 8   NDVI               1982 non-null   float64
 9   NDWI               1982 non-null   float64
 10  NBR                1982 non-null   float64
 11  DEM                1982 non-null   float64
 12  aspect             1982 non-null   int64  
 13  hillshade          1982 non-null   int64  
 14  slope              1982 non-null   int64  
 15  bi                 1982 non-null   float64
 16  erc                1982 

In [ ]:
fires = fires_df.drop(columns=["Unnamed: 0"])
fires.head(50)

,FIRE_ID,UrbanAngle,occur_id,point_id,IsUrban,WUIBreach,Month,NDVI,NDWI,NBR,DEM,aspect,hillshade,slope,bi,erc,eto,fm100,fm1000,pr,rmax,rmin,th,pop_density,tmmn,tmmx,vpd,vs,LandCover,Y,Water,Very_low_rural,Low_density_rural,Rural_cluster,Suburban,Semi_dense_urban,Dense_urban,Urban_centre,first_urban_point,RemoveFlag
0,CA3622612010420250902,-163.647115,1,0,0,0,9,0.141225,0.178077,0.019172,102.352814,56,180,0,62.0,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,0.000000,293.0,313.5,3.75,2.7,81,1,0,1,0,0,0,0,0,0,60,False
1,CA3622612010420250902,-163.647115,1,1,0,0,9,0.111281,0.156092,-0.058161,102.741325,19,180,0,62.0,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,0.000000,293.0,313.5,3.75,2.7,81,1,0,1,0,0,0,0,0,0,60,False
2,CA3622612010420250902,-163.647115,1,2,0,0,9,0.130974,0.176366,-0.006227,102.741325,19,180,0,62.0,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,0.000000,293.0,313.5,3.75,2.7,82,1,0,1,0,0,0,0,0,0,60,False
3,CA3622612010420250902,-163.647115,1,3,0,0,9,0.131013,0.176502,0.007718,103.105774,51,179,0,62.0,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,0.000000,293.0,313.5,3.75,2.7,82,1,0,1,0,0,0,0,0,0,60,False
4,CA3622612010420250902,-163.647115,1,4,0,0,9,0.099985,0.143464,-0.029291,103.281387,341,181,0,62.0,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,0.000000,293.0,313.5,3.75,2.7,82,1,0,1,0,0,0,0,0,0,60,False
5,CA3622612010420250902,-163.647115,1,5,0,0,9,0.082638,0.132177,0.007172,103.392738,147,180,0,62.0,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,0.000000,293.0,313.5,3.75,2.7,82,1,0,1,0,0,0,0,0,0,60,False
6,CA3622612010420250902,-163.647115,1,6,0,0,9,0.063996,0.113926,-0.042081,103.392738,147,180,0,62.0,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,0.175505,293.0,313.5,3.75,2.7,82,1,0,1,0,0,0,0,0,0,60,False
7,CA3622612010420250902,-163.647115,1,7,0,0,9,0.107337,0.153805,-0.036135,103.047516,179,180,0,62.0,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,0.175505,293.0,313.5,3.75,2.7,82,1,0,1,0,0,0,0,0,0,60,False
8,CA3622612010420250902,-163.647115,1,8,0,0,9,0.135910,0.170315,0.005047,103.047516,179,180,0,62.0,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,0.175505,293.0,313.5,3.75,2.7,21,1,0,1,0,0,0,0,0,0,60,False
9,CA3622612010420250902,-163.647115,1,9,0,0,9,0.100362,0.148615,-0.052423,102.941162,250,181,0,62.0,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,0.175505,293.0,313.5,3.75,2.7,21,1,0,1,0,0,0,0,0,0,60,False


In [ ]:
def create_fire_id_groups(df):
    df['FIRE_ID'] = df['FIRE_ID'].astype(str)
    df = df.reset_index(drop=True)
    fire_groups = df.groupby('FIRE_ID')
    fire_list = []
    for fire_id, fire_seq in fire_groups:
      fire_list.append(fire_seq)
    return fire_list

def create_sequences_from_fire_ids(list_of_fires):
    no_sequences = 0
    max_seq = 0
    min_seq = 100
    longest_id = 0
    X_list = []
    Y_list = []
    for df in list_of_fires:
      df_groups = df.groupby('occur_id')
      for idx, seq in df_groups:
          seq_len = len(seq)
          no_sequences += 1
          if seq_len > max_seq:
              max_seq = seq_len
              longest_id = seq["occur_id"].iloc[0]
          if seq_len < min_seq:
              min_seq = seq_len

          Y = seq["Y"]
          Y_list.append(Y)
          X = seq.drop(columns=["LandCover", "WUIBreach", "Month", "first_urban_point", "RemoveFlag"])#.values
          X_list.append(X)

    return no_sequences, max_seq, min_seq, longest_id, X_list, Y_list

In [ ]:
def create_sequences_from_fire_ids_twice(list_of_fires):
    import gc
    no_sequences = 0
    max_seq = 0
    min_seq = 100
    longest_id = 0
    X_list = []
    Y_list = []
    for df in list_of_fires:

      df_odd = df[df['point_id'] % 2 != 0]
      df = df[df['point_id'] % 2 != 1]

      for idx, seq in df_odd.groupby('occur_id'):
          seq_len = len(seq)
          no_sequences += 1
          if seq_len > max_seq:
              max_seq = seq_len
              longest_id = seq["occur_id"].iloc[0]
          if seq_len < min_seq:
              min_seq = seq_len

          Y = seq["Y"]
          Y_list.append(Y)
          X = seq.drop(columns=["LandCover", "WUIBreach", "Month", "first_urban_point", "RemoveFlag"])#.values
          X_list.append(X)

      for idx, seq in df.groupby('occur_id'):
            seq_len = len(seq)
            no_sequences += 1
            if seq_len > max_seq:
                max_seq = seq_len
                longest_id = seq["occur_id"].iloc[0]
            if seq_len < min_seq:
                min_seq = seq_len

            Y = seq["Y"]
            Y_list.append(Y)
            X = seq.drop(columns=["LandCover", "WUIBreach", "Month", "first_urban_point", "RemoveFlag"])#.values
            X_list.append(X)

    return no_sequences, max_seq, min_seq, longest_id, X_list, Y_list

In [ ]:
def create_sequences_for_training(df):
    df_groups = df.groupby('occur_id')
    no_sequences = 0
    max_seq = 0
    min_seq = 100
    longest_id = 0
    X_list = []
    Y_list = []
    #y_last = []
    for idx, seq in df_groups:
        seq_len = len(seq)
        no_sequences += 1
        if seq_len > max_seq:
            max_seq = seq_len
            longest_id = seq["occur_id"].iloc[0]
        if seq_len < min_seq:
            min_seq = seq_len

        Y = seq["Y"]
        Y_list.append(Y)
        X = seq.drop(columns=["LandCover", "WUIBreach", "Month", "first_urban_point", "RemoveFlag"])
        X_list.append(X)

    return no_sequences, max_seq, min_seq, longest_id, X_list, Y_list


In [ ]:
def scaler_transform(X_list, scaler):
    new_Xlist = []
    for X_sequence in X_list:
        X_sequence = X_sequence.drop(columns=["IsUrban", "occur_id", "Y", "point_id", "FIRE_ID"]) #'Y'
        X_values = X_sequence.values
        X_norm = scaler.transform(X_values)
        new_Xlist.append(X_norm)
    return new_Xlist

In [ ]:
all_fires_list = create_fire_id_groups(fires)
#X_train_list, X_test_list = train_test_split(all_fires_list, test_size=0.3, train_size=0.7, random_state=19, shuffle=True, stratify=None)
X_test_list = all_fires_list

In [ ]:
#print(f"Training fires: {len(X_train_list)}")
print(f"Testing fires: {len(X_test_list)}")

Testing fires: 5


In [ ]:
#_, steps, _, _, X_train, Y_train = create_sequences_from_fire_ids_twice(X_train_list)
_, _, _, _, X_test, Y_test = create_sequences_from_fire_ids_twice(X_test_list)

In [ ]:
print(X_test[0])

                  FIRE_ID  UrbanAngle  occur_id  point_id  IsUrban      NDVI  \
1   CA3622612010420250902 -163.647115         1         1        0  0.111281   
3   CA3622612010420250902 -163.647115         1         3        0  0.131013   
5   CA3622612010420250902 -163.647115         1         5        0  0.082638   
7   CA3622612010420250902 -163.647115         1         7        0  0.107337   
9   CA3622612010420250902 -163.647115         1         9        0  0.100362   
11  CA3622612010420250902 -163.647115         1        11        0  0.092261   
13  CA3622612010420250902 -163.647115         1        13        0  0.093276   
15  CA3622612010420250902 -163.647115         1        15        0  0.105476   
17  CA3622612010420250902 -163.647115         1        17        0  0.090078   
19  CA3622612010420250902 -163.647115         1        19        0  0.103602   
21  CA3622612010420250902 -163.647115         1        21        0  0.057049   
23  CA3622612010420250902 -163.647115   

In [ ]:
X_test[0].columns

Index(['FIRE_ID', 'UrbanAngle', 'occur_id', 'point_id', 'IsUrban', 'NDVI',
       'NDWI', 'NBR', 'DEM', 'aspect', 'hillshade', 'slope', 'bi', 'erc',
       'eto', 'fm100', 'fm1000', 'pr', 'rmax', 'rmin', 'th', 'pop_density',
       'tmmn', 'tmmx', 'vpd', 'vs', 'Y', 'Water', 'Very_low_rural',
       'Low_density_rural', 'Rural_cluster', 'Suburban', 'Semi_dense_urban',
       'Dense_urban', 'Urban_centre'],
      dtype='object')

Index(['FIRE_ID', 'UrbanAngle', 'occur_id', 'point_id', 'IsUrban', 'NDVI',
       'NDWI', 'NBR', 'DEM', 'aspect', 'hillshade', 'slope', 'bi', 'erc',
       'eto', 'fm100', 'fm1000', 'pr', 'rmax', 'rmin', 'th', 'pop_density',
       'tmmn', 'tmmx', 'vpd', 'vs', 'Y', 'Water', 'Very_low_rural',
       'Low_density_rural', 'Rural_cluster', 'Suburban', 'Semi_dense_urban',
       'Dense_urban', 'Urban_centre'],
      dtype='object')

In [ ]:
import pickle
model = keras.saving.load_model('/content/drive/MyDrive/wildfire_model_WUI_100m.keras')
X_scaler = pickle.load(open('/content/drive/MyDrive/wildfire_scaler_WUI.keras', 'rb'))

In [ ]:
import copy
X_test_ref = []
for df in X_test:
  df_ref = df.copy(deep=True)
  X_test_ref.append(df_ref)

len(X_test_ref)

24

In [ ]:
urb_list = []
for df in X_test_ref:
  X_urb = df["IsUrban"]
  urb_list.append(X_urb)

len(urb_list)

24

In [ ]:
X_test = scaler_transform(X_test, X_scaler)
X_test_padded = sequence.pad_sequences(X_test, padding='post', dtype='float32', value=-9)
X_test_ref_padded = sequence.pad_sequences(X_test, padding='post', dtype='float32', value=-9)
Y_test_padded = sequence.pad_sequences(Y_test, padding='post', dtype='float32', value=-9)
X_test_IsUrban_padded = sequence.pad_sequences(urb_list, padding='post', dtype='float32', value=-9)

In [ ]:
print(X_test_padded.shape)
print(Y_test_padded.shape)
print(X_test_IsUrban_padded.shape)
print(X_test_ref_padded.shape)

(24, 101, 30)
(24, 101)
(24, 101)
(24, 101, 30)


In [ ]:
# https://machinelearningmastery.com/multivariate-time-series-forecasting-lstms-keras/

# make a prediction
yhat = model.predict(X_test_padded)
threshold = 0.4



1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


In [ ]:
y_pred = np.where(yhat > threshold, 1,0).astype(int)
y = Y_test_padded.astype(int)

In [ ]:
y_pred = y_pred.reshape(y_pred.shape[0], y_pred.shape[1])

print(y_pred.shape)
print(y.shape)

(24, 101)
(24, 101)


In [ ]:
X_test_IsUrban_padded = X_test_IsUrban_padded.astype(int)
urb = X_test_IsUrban_padded.reshape(X_test_IsUrban_padded.shape[0], X_test_IsUrban_padded.shape[1])
print(urb.shape)

(24, 101)


In [ ]:
print(y[0])

[ 1  1  1  1  1  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9
 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9
 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9
 -9 -9 -9 -9 -9]


In [ ]:
print(y_pred[0])

[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [ ]:
print(urb[3])

[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  1  1 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9
 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9 -9
 -9 -9 -9 -9 -9]


In [ ]:
y_list = []
y_pred_list = []
y_final = []
y_pred_final = []
y_final_lengths = []
y_list_urb = []
y_pred_list_urb = []

for y_seq, y_pred_seq, urb_seq in zip(y, y_pred, urb):
  seq_mask = y_seq != -9
  y_seq = y_seq[seq_mask]
  y_pred_seq = y_pred_seq[seq_mask]
  urb_mask_int = urb_seq[seq_mask]

  y_urb_seq = y_seq[urb_mask_int.astype(bool)]
  y_pred_urb_seq = y_pred_seq[urb_mask_int.astype(bool)]

  if len(y_urb_seq) > 0:
    y_WUI = np.max(y_urb_seq)
    y_WUI_pred = np.max(y_pred_urb_seq)
  else:
    y_WUI = 0
    y_WUI_pred = 0

  y_final.append([y_WUI])
  y_pred_final.append([y_WUI_pred])

  y_list.append(y_seq)
  y_pred_list.append(y_pred_seq)
  y_list_urb.append(y_urb_seq)
  y_pred_list_urb.append(y_pred_urb_seq)

  y_urb_values = y_seq[urb_mask_int.astype(bool)]
  y_pred_urb_values = y_pred_seq[urb_mask_int.astype(bool)]


  if np.any(urb_seq == 1):
    first_urban = np.where(urb_seq == 1)[0][0]
    y_final_lengths.append(first_urban + 1)
  else:
    y_final_lengths.append(-9)

In [ ]:
X_test_output_list = []
for test_df, y_t in zip(X_test_ref, y_pred_list):
  test_df_copy = test_df.copy()
  test_df_copy["Y_predicted"] = y_t
  X_test_output_list.append(test_df_copy)

print(X_test_output_list[0].head())

                 FIRE_ID  UrbanAngle  occur_id  point_id  IsUrban      NDVI  \
1  CA3622612010420250902 -163.647115         1         1        0  0.111281   
3  CA3622612010420250902 -163.647115         1         3        0  0.131013   
5  CA3622612010420250902 -163.647115         1         5        0  0.082638   
7  CA3622612010420250902 -163.647115         1         7        0  0.107337   
9  CA3622612010420250902 -163.647115         1         9        0  0.100362   

       NDWI       NBR         DEM  aspect  hillshade  slope    bi   erc  eto  \
1  0.156092 -0.058161  102.741325      19        180      0  62.0  84.0  7.4   
3  0.176502  0.007718  103.105774      51        179      0  62.0  84.0  7.4   
5  0.132177  0.007172  103.392738     147        180      0  62.0  84.0  7.4   
7  0.153805 -0.036135  103.047516     179        180      0  62.0  84.0  7.4   
9  0.148615 -0.052423  102.941162     250        181      0  62.0  84.0  7.4   

   fm100  fm1000   pr       rmax  rmin    th

In [ ]:
out_df = X_test_output_list[0]
for df in X_test_output_list[1:]:
  out_df = pd.concat([out_df, df])


In [ ]:
len(out_df)

1982

In [ ]:
print(X_test_padded.shape)

(24, 101, 30)


In [ ]:
out_df.to_csv('/content/drive/MyDrive/wildfire_model_df_testing.csv')

In [ ]:
print(y_final_lengths[:100])


[np.int64(31), np.int64(31), np.int64(38), np.int64(49), np.int64(38), np.int64(50), np.int64(94), np.int64(94), np.int64(90), -9, np.int64(91), np.int64(99), np.int64(83), np.int64(90), np.int64(101), np.int64(91), np.int64(99), np.int64(83), np.int64(95), np.int64(92), np.int64(95), np.int64(95), np.int64(93), np.int64(95)]


In [ ]:
print(y_list[0])
print("\n")
print(y_pred_list[0])


[1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0]


In [ ]:
print(y_list_urb[1])
print("\n")
print(y_pred_list_urb[1])

[0 0 0]


[0 0 0]


In [ ]:
print(y_final)
print("\n")
print(y_pred_final)
print("\n")
print(y_final_lengths)

[[np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [0], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)]]


[[np.int64(0)], [np.int64(0)], [np.int64(1)], [np.int64(1)], [np.int64(1)], [np.int64(1)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [0], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)], [np.int64(0)]]


[np.int64(31), np.int64(31), np.int64(38), np.int64(49), np.int64(38), np.int64(50), np.int64(94), np.int64(94), np.int64(90), -9, np.int64(91), np.int64(99), np.int64(83), np.int64(90), np.int64(101), np.int64(91), np.int64(99), np.int64(83), np.int64(95), np.int64(92), np.int64(95), np.int64

In [ ]:
def calculate_metrics_all_points(ytrue_list, yhat_list, min_length, max_length, Breach=False):
  TP_count = 0
  FP_count = 0
  TN_count = 0
  FN_count = 0
  for y_seq, y_pred_seq in zip(ytrue_list, yhat_list):
    #if not Breach:
    if len(y_seq) > min_length and len(y_seq) <= max_length:
      for i in range(len(y_seq)):
        if y_seq[i] == 1 and y_pred_seq[i] == 1:
          TP_count += 1
        elif y_seq[i] == 0 and y_pred_seq[i] == 1:
          FP_count += 1
        elif y_seq[i] == 1 and y_pred_seq[i] == 0:
          FN_count += 1
        elif y_seq[i] == 0 and y_pred_seq[i] == 0:
          TN_count += 1

    total_rows = TP_count + FP_count + TN_count + FN_count
    try:
      accuracy = round((TP_count + TN_count) / total_rows, 2)
    except:
      accuracy = 0

    # precision tp / (tp + fp)
    try:
      precision = round(TP_count / (TP_count + FP_count),2)
    except:
      precision = 0

    # recall: tp / (tp + fn)
    try:
      recall = round(TP_count / (TP_count + FN_count), 2)
    except:
      recall = 0

    # f1: 2 tp / (2 tp + fp + fn)
    try:
      f1 = round(2 * ((precision * recall)/ (precision + recall)), 2)
    except:
      f1 = 0


    confusion_matrix = [TP_count, FP_count, FN_count, TN_count]


  return accuracy, precision, recall, f1, confusion_matrix


In [ ]:
def test_calculate_metrics_all_points(ytrue_list, yhat_list, min_distance, max_distance, Breach=False):
  TP_count = 0
  FP_count = 0
  TN_count = 0
  FN_count = 0
  for y_seq, y_pred_seq in zip(ytrue_list, yhat_list):
    for i in range(len(y_seq)):
      if i >= min_distance and i < max_distance:
        if y_seq[i] == 1 and y_pred_seq[i] == 1:
          TP_count += 1
        elif y_seq[i] == 0 and y_pred_seq[i] == 1:
          FP_count += 1
        elif y_seq[i] == 1 and y_pred_seq[i] == 0:
          FN_count += 1
        elif y_seq[i] == 0 and y_pred_seq[i] == 0:
          TN_count += 1

    total_rows = TP_count + FP_count + TN_count + FN_count
    try:
      accuracy = round((TP_count + TN_count) / total_rows, 2)
    except:
      accuracy = 0

    # precision tp / (tp + fp)
    try:
      precision = round(TP_count / (TP_count + FP_count),2)
    except:
      precision = 0

    # recall: tp / (tp + fn)
    try:
      recall = round(TP_count / (TP_count + FN_count), 2)
    except:
      recall = 0

    # f1: 2 tp / (2 tp + fp + fn)
    try:
      f1 = round(2 * ((precision * recall)/ (precision + recall)), 2)
    except:
      f1 = 0


    confusion_matrix = [TP_count, FP_count, FN_count, TN_count]


  return accuracy, precision, recall, f1, confusion_matrix

In [ ]:
model_accuracy, model_precision, model_recall, model_f1, model_confusion = calculate_metrics_all_points(y_list, y_pred_list, 0, X_test_padded.shape[1])
print(f"Accuracy: {model_accuracy}")
print(f"Recall: {model_recall}")
print(f"Precision: {model_precision}")
print(f"F1 score: {model_f1}")
print(f"Confusion matrix: {model_confusion}")

Accuracy: 0.79
Recall: 0.83
Precision: 0.37
F1 score: 0.51
Confusion matrix: [220, 374, 45, 1343]


In [ ]:
urban_fire_accuracy, urban_fire_precision, urban_fire_recall, urban_fire_f1, urban_fire_confusion = calculate_metrics_all_points(y_list_urb, y_pred_list_urb, 0, X_test_padded.shape[1])

# this is a fire / non fire prediction, not a WUI breach prediction
# for urban points, did the fire spread there?

print(f"Accuracy (urban fire): {urban_fire_accuracy}")
print(f"Recall (urban fire): {urban_fire_recall}")
print(f"Precision (urban fire): {urban_fire_precision}")
print(f"F1 score (urban fire): {urban_fire_f1}")
print(f"Confusion matrix (urban fire): {urban_fire_confusion}")

Accuracy (urban fire): 0.64
Recall (urban fire): 0
Precision (urban fire): 0.0
F1 score (urban fire): 0
Confusion matrix (urban fire): [0, 32, 0, 56]


In [ ]:
WUI_breach_accuracy, WUI_breach_precision, WUI_breach_recall, WUI_breach_f1, WUI_breach_confusion = calculate_metrics_all_points(y_final, y_pred_final, 0, X_test_padded.shape[1])

print(f"WUI Breach Accuracy: {WUI_breach_accuracy}")
print(f"WUI Breach Recall: {WUI_breach_recall}")
print(f"WUI Breach Precision: {WUI_breach_precision}")
print(f"WUI Breach F1 score: {WUI_breach_f1}")
print(f"WUI Breach Confusion matrix: {WUI_breach_confusion}")

WUI Breach Accuracy: 0.83
WUI Breach Recall: 0
WUI Breach Precision: 0.0
WUI Breach F1 score: 0
WUI Breach Confusion matrix: [0, 4, 0, 20]


In [ ]:
def display_confusion_matrix(confusion_matrix, min_distance, max_distance):
  header = f"{min_distance}km to {max_distance}km"
  c_f = pd.DataFrame(columns=[header, 'True', 'False'])

  new_row_positive = pd.DataFrame([["Positive", confusion_matrix[0], confusion_matrix[1]]], columns=c_f.columns)
  c_f = pd.concat([c_f, new_row_positive], ignore_index =True)

  new_row_negative = pd.DataFrame([["Negative", confusion_matrix[3], confusion_matrix[2]]], columns=c_f.columns)
  c_f = pd.concat([c_f, new_row_negative], ignore_index =True)

  return c_f


In [ ]:
def calculate_metrics_by_distance(ytrue_list, yhat_list, interval):
  min_ = 0
  start_ = 0 + interval
  max_ = start_
  all_cm = []
  all_point_stats = pd.DataFrame(columns=['Min_Kilometres', 'Max_Kilometres', 'Accuracy', 'Precision', 'Recall', 'F1'])
  for max_ in range(start_, 101, interval):
    accuracy, precision, recall, f1, conf = test_calculate_metrics_all_points(ytrue_list, yhat_list, min_, max_)
    new_row = pd.DataFrame([[(min_/10), (max_/10), accuracy, precision, recall, f1]], columns=all_point_stats.columns)
    all_point_stats = pd.concat([all_point_stats, new_row], ignore_index =True)
    conf_matrix = display_confusion_matrix(conf, min_/10, max_/10)
    all_cm.append(conf_matrix)
    min_ += interval
    max_ += interval

  return all_point_stats, all_cm


In [ ]:
def calculate_WUI_breach_metrics_by_distance(ytrue_list, yhat_list, y_length_list, interval):
  min_ = 0
  start_ = 0 + interval
  max_ = start_
  all_cm = []
  all_point_stats = pd.DataFrame(columns=['Min_Kilometres', 'Max_Kilometres', 'Accuracy', 'Precision', 'Recall', 'F1'])

  for max_ in range(start_, 105, interval):
    y_check = []
    y_pred_check = []
    for y, y_pred, y_length in zip(ytrue_list, yhat_list, y_length_list):
      if y_length > min_ and y_length <= max_:
        y_check.append(y)
        y_pred_check.append(y_pred)
    accuracy, precision, recall, f1, conf = test_calculate_metrics_all_points(y_check, y_pred_check, 0, X_test_padded.shape[1])
    new_row = pd.DataFrame([[(min_/10), (max_/10), accuracy, precision, recall, f1]], columns=all_point_stats.columns)
    all_point_stats = pd.concat([all_point_stats, new_row], ignore_index =True)
    conf_matrix = display_confusion_matrix(conf, min_/10, max_/10)
    all_cm.append(conf_matrix)
    min_ += interval
    max_ += interval

  return all_point_stats, all_cm


In [ ]:
distance_stats, distance_cm = calculate_metrics_by_distance(y_list, y_pred_list, 10)

print(distance_stats)
print("\n")
for c in distance_cm:
  print(c)
  print("\n")

   Min_Kilometres  Max_Kilometres  Accuracy  Precision  Recall    F1
0             0.0             1.0      0.79       0.79    0.99  0.88
1             1.0             2.0      0.27       0.17    0.55  0.26
2             2.0             3.0      0.57       0.00    0.00  0.00
3             3.0             4.0      0.80       0.00    0.00  0.00
4             4.0             5.0      0.82       0.00    0.00  0.00
5             5.0             6.0      0.98       0.00    0.00  0.00
6             6.0             7.0      1.00       0.00    0.00  0.00
7             7.0             8.0      1.00       0.00    0.00  0.00
8             8.0             9.0      1.00       0.00    0.00  0.00
9             9.0            10.0      1.00       0.00    0.00  0.00


  0.0km to 1.0km True False
0       Positive  189    49
1       Negative    0     2


  1.0km to 2.0km True False
0       Positive   31   151
1       Negative   33    25


  2.0km to 3.0km True False
0       Positive    0    86
1       Neg

/tmp/ipykernel_577/2887613809.py:10: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_point_stats = pd.concat([all_point_stats, new_row], ignore_index =True)


In [ ]:
breach_distance_stats, breach_distance_cm = calculate_WUI_breach_metrics_by_distance(y_final, y_pred_final, y_final_lengths, 25)

print(breach_distance_stats)
print("\n")
for c in breach_distance_cm:
  print(c)
  print("\n")

UnboundLocalError: cannot access local variable 'accuracy' where it is not associated with a value

In [ ]:
all_confusion_matrix = display_confusion_matrix(model_confusion, 0, 100)
print(all_confusion_matrix)

  0km to 100km  True False
0     Positive   220   374
1     Negative  1343    45
